# Investigación para hacer funcionar agentes de LangChain

In [54]:
# Importaciones necesarias
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain_openai import AzureChatOpenAI
from langchain.schema import SystemMessage
import os
from dotenv import load_dotenv

# Cargar variables de entorno
load_dotenv('../.env')

# Configurar las credenciales de Azure OpenAI
#AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
#os.environ["AZURE_OPENAI_ENDPOINT"] = os.getenv("AZURE_OPENAI_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME")
OPENAI_API_VERSION =os.getenv("OPENAI_API_VERSION")  # Ajusta según la versión que uses

# Inicializar el modelo de Azure OpenAI
llm = AzureChatOpenAI(
    deployment_name=DEPLOYMENT_NAME,  # Ajusta según tu deployment
    model_name="gpt-4o-mini",       # Ajusta según tu modelo
    temperature=0.7,
    verbose=True
)

# Cargar herramientas básicas
tools = load_tools(["llm-math"], llm=llm)

# Inicializar el agente
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,

)

print(type(agent))
# # Ejemplo de uso
# def test_agent():
#     response = agent.run(
#         "¿Cuál es la raíz cuadrada de 256 multiplicada por 3? La respuesta final debe ser en español.",
#     )
#     print("Respuesta del agente:", response)

# test_agent()

<class 'langchain.agents.agent.AgentExecutor'>


C:\Users\SoporteTI\AppData\Local\Temp\ipykernel_21848\1983670847.py:29: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


# Funciones personalizadas

In [2]:
# Importaciones necesarias
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain_openai import AzureChatOpenAI
from langchain.schema import SystemMessage
from langchain.tools import BaseTool
import os
from dotenv import load_dotenv
from typing import Type, Optional

# Cargar variables de entorno
load_dotenv('../.env')

# Configurar las credenciales de Azure OpenAI
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME")
OPENAI_API_VERSION = os.getenv("OPENAI_API_VERSION")

# Definir la herramienta FizzBuzz
class FizzBuzzTool(BaseTool):
    name: str = "fizz_buzz"
    description: str = """Útil para jugar FizzBuzz con un número.
    La herramienta aplica las reglas de FizzBuzz:
    - Si el número es divisible por 3, devuelve 'Fizz'
    - Si el número es divisible por 5, devuelve 'Buzz'
    - Si el número es divisible por ambos, devuelve 'FizzBuzz'
    - En cualquier otro caso, devuelve el número como string
    El input debe ser un número entero positivo."""

    def _run(self, numero: int) -> str:
        try:
            num = int(numero)
            if num <= 0:
                return "Por favor, proporciona un número entero positivo."
            
            resultado = ""
            if num % 3 == 0:
                resultado += "Fizz"
            if num % 5 == 0:
                resultado += "Buzz"
            
            return resultado if resultado else str(num)
        
        except ValueError:
            return "Error: Por favor, proporciona un número entero válido."

    def _arun(self, numero: int):
        raise NotImplementedError("FizzBuzz no soporta ejecución asíncrona")

# Inicializar el modelo de Azure OpenAI
llm = AzureChatOpenAI(
    deployment_name=DEPLOYMENT_NAME,
    model_name="gpt-4o-mini",
    temperature=0.7,
    verbose=True
)

# Cargar herramientas
tools = load_tools(["llm-math"], llm=llm)
tools.append(FizzBuzzTool())  # Agregar nuestra herramienta personalizada

# Inicializar el agente
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)

# Raíz cuadrada
response = agent.run(
    "¿Cuál es la raíz cuadrada de 256 multiplicada por 3? La respuesta final debe ser en español.",
)
print("Respuesta del agente (matemáticas):", response)


# FizzBuzz
number = 15
response = agent.run(
    f"¿Qué resultado da FizzBuzz para el número {number}? Explica por qué.",
)
print("Respuesta del agente (FizzBuzz):", response)





> Entering new AgentExecutor chain...
I need to calculate the square root of 256 and then multiply it by 3. I can use the Calculator tool for this.
Action: Calculator
Action Input: sqrt(256) * 3
Observation: Answer: 48.0
Thought:I need to translate the answer to Spanish.
Action: N/A
Action Input: N/A
Observation: N/A is not a valid tool, try one of [Calculator, fizz_buzz].
Thought:I can simply write the answer in Spanish.
Final Answer: La raíz cuadrada de 256 multiplicada por 3 es 48.0.

> Finished chain.
Respuesta del agente (matemáticas): La raíz cuadrada de 256 multiplicada por 3 es 48.0.


> Entering new AgentExecutor chain...
I need to apply the rules of FizzBuzz to the number 15 to determine the result.
Action: fizz_buzz
Action Input: 15
Observation: FizzBuzz
Thought:15 es divisible por 3 y por 5, por lo que FizzBuzz devuelve "FizzBuzz".
Thought: That answers the original question.
Final Answer: FizzBuzz.

> Finished chain.
Respuesta del agente (FizzBuzz): FizzBuzz.


# Pandas Agent Simple

In [3]:
# Importaciones necesarias
from langchain_openai import AzureChatOpenAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain.agents import AgentType
import pandas as pd
from dotenv import load_dotenv
import os

# Cargar variables de entorno
load_dotenv('../.env')

# Configurar las credenciales de Azure OpenAI
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME")
OPENAI_API_VERSION = os.getenv("OPENAI_API_VERSION")

# Crear un DataFrame de ejemplo
df = pd.read_csv("../assets/metro.csv")

#df.set_index('fecha', inplace=True)
print("DataFrame de ejemplo:")
print(df)

# Inicializar el modelo
llm = AzureChatOpenAI(
    deployment_name=DEPLOYMENT_NAME,
    model_name="gpt-4o-mini",
    temperature=0,  # Usar temperatura 0 para respuestas más consistentes
    verbose=True
)

# Crear el agente de Pandas
agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    allow_dangerous_code=True,
)

def test_agent_capabilities():
    """Función para probar diferentes capacidades del agente"""
    
    # 1. Análisis básico
    print("1. Análisis básico:")
    response = agent.invoke("¿Cuántas filas y columnas tiene el DataFrame?")
    print(f"Respuesta: {response}\n")
    
    # 2. Agregaciones
    print("2. Agregaciones:")
    response = agent.invoke("¿Cuál es el promedio de pasajeros en total?")
    print(f"Respuesta: {response}\n")
    
    # 3. Filtrado y condiciones
    print("3. Filtrado:")
    response = agent.invoke("En total, ¿Cuántos pasajeros se tienen registrados?")
    print(f"Respuesta: {response}\n")
    
    # 4. Análisis temporal
    print("4. Análisis temporal:")
    response = agent.invoke("¿Cuál fue el día con mayor afluencia de pasajeros?")
    print(f"Respuesta: {response}\n")
    
    # 5. Análisis estadístico
    print("5. Estadísticas:")
    response = agent.invoke("Dame un resumen estadístico del uso del metro (cantidad de pasajeros)")
    print(f"Respuesta: {response}\n")

test_agent_capabilities()

DataFrame de ejemplo:
         mes  pasajeros
0    1990/01      128.3
1    1990/02      113.0
2    1990/03      126.7
3    1990/04      109.5
4    1990/05      123.5
..       ...        ...
414  2024/07       95.5
415  2024/08      100.9
416  2024/09       99.8
417  2024/10      109.8
418  2024/11      103.4

[419 rows x 2 columns]
1. Análisis básico:


> Entering new AgentExecutor chain...
Thought: We can use the `shape` attribute of the dataframe to get the number of rows and columns.
Action: python_repl_ast
Action Input: df.shape(419, 2)The dataframe has 419 rows and 2 columns.
Final Answer: The dataframe has 419 rows and 2 columns.

> Finished chain.
Respuesta: {'input': '¿Cuántas filas y columnas tiene el DataFrame?', 'output': 'The dataframe has 419 rows and 2 columns.'}

2. Agregaciones:


> Entering new AgentExecutor chain...
Thought: I need to calculate the mean of the 'pasajeros' column in the dataframe
Action: python_repl_ast
Action Input: df['pasajeros'].mean()116.536992840

# Pandas Agent con un gestor de Errores

In [5]:
from langchain_openai import AzureChatOpenAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain.agents import AgentType
import pandas as pd
from dotenv import load_dotenv
import os
load_dotenv('../.env')
llm = AzureChatOpenAI(
    deployment_name=os.getenv("DEPLOYMENT_NAME"),
    model_name="gpt-4o-mini",
    temperature=0  # Usar temperatura 0 para respuestas más consistentes
)

agent = create_pandas_dataframe_agent(
    llm,
    df,  
    verbose=True,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,  # Maneja errores de parsing
    max_iterations=3,  # Limita el número de intentos
    early_stopping_method="generate",  # Método de detención temprana
    allow_dangerous_code=True,  # Permite código peligroso
)

# Función de prueba con manejo de errores
def safe_agent_query(question: str, max_retries: int = 2):
    """
    Ejecuta una consulta al agente con manejo de errores y reintentos
    """
    for attempt in range(max_retries):
        try:
            response = agent.run(question)
            return response
        except Exception as e:
            if attempt == max_retries - 1:  # Si es el último intento
                return f"Error en la consulta después de {max_retries} intentos: {str(e)}"
            print(f"Intento {attempt + 1} falló, reintentando...")
            continue

# Ejemplo de uso
if __name__ == "__main__":
    # Probar con tu consulta
    resultado = safe_agent_query("¿Qué día tuvo el mayor número de pasajeros?")
    print(resultado)
    
    # También puedes probar una consulta más específica
    resultado_alternativo = safe_agent_query("""
    Por favor:
    1. Encuentra el valor máximo en la columna de pasajeros
    2. Identifica la fecha correspondiente a ese valor
    3. Muestra ambos datos
    """)
    print(resultado_alternativo)

c:\Users\SoporteTI\Documents\project_directory\GitHubRepos\de_avatar_lowpoly\.venv\Lib\site-packages\langchain_experimental\agents\agent_toolkits\pandas\base.py:283: UserWarning: Received additional kwargs {'handle_parsing_errors': True} which are no longer supported.
  warnings.warn(
C:\Users\SoporteTI\AppData\Local\Temp\ipykernel_17044\3537123871.py:38: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = agent.run(question)




> Entering new AgentExecutor chain...
Thought: We need to find the row with the highest value in the "pasajeros" column.
Action: Use the `idxmax()` method to find the index of the row with the highest value in the "pasajeros" column.
Action Input: `df['pasajeros'].idxmax()`Use the `idxmax()` method to find the index of the row with the highest value in the "pasajeros" column. is not a valid tool, try one of [python_repl_ast].I need to use the `python_repl_ast` tool to execute the command.
Action: Use the `python_repl_ast` tool to execute the command.
Action Input: `df['pasajeros'].idxmax()`Use the `python_repl_ast` tool to execute the command. is not a valid tool, try one of [python_repl_ast].I need to use the `python_repl_ast` tool to execute the command.
Action: Use the `python_repl_ast` tool to execute the command.
Action Input: `df['pasajeros'].idxmax()`Use the `python_repl_ast` tool to execute the command. is not a valid tool, try one of [python_repl_ast].Intento 1 falló, reinte

In [6]:
resultado_alternativo = safe_agent_query("""
    Por favor:
    1. Encuentra el valor máximo en la columna de pasajeros
    2. Identifica la fecha correspondiente a ese valor
    3. Muestra ambos datos
    """)



> Entering new AgentExecutor chain...
Thought: We need to find the row with the maximum value in the 'pasajeros' column and then get the values of both 'mes' and 'pasajeros' columns for that row.
Action: python_repl_ast
Action Input:
```
max_row = df.loc[df['pasajeros'].idxmax()]
max_row[['mes', 'pasajeros']]
```mes          2013/10
pasajeros      154.7
Name: 285, dtype: objectWe found the row with the maximum value in the 'pasajeros' column and got the values of both 'mes' and 'pasajeros' columns for that row.
Final Answer: mes: 2013/10, pasajeros: 154.7

> Finished chain.


# Agente de pandas que ejecuta el código

In [5]:
from langchain_openai import AzureChatOpenAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain.agents import AgentType
import pandas as pd
from typing import Tuple, Optional
import ast

class InteractiveDataFrameAgent:
    def __init__(self, llm, df: pd.DataFrame):
        self.df = df
        self.agent = create_pandas_dataframe_agent(
            llm,
            self.df,
            verbose=True,
            agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
            allow_dangerous_code=True,
            handle_parsing_errors=True,
            max_iterations=3,
            early_stopping_method="generate"
        )
        
    def process_query(self, query: str) -> Tuple[str, Optional[pd.DataFrame]]:
        """
        Processes a query and returns both the response and modified DataFrame if applicable
        """
        try:
            # Add explicit instruction to return modifications
            enhanced_query = f"""
            {query}
            If this requires modifying the DataFrame, please:
            1. Explain the modifications
            2. Return the Python code needed to make these modifications
            Format any code modifications as: EXEC_CODE: <your_code_here>
            """
            
            response = self.agent.run(enhanced_query)
            
            # Check if response contains executable code
            if "EXEC_CODE:" in response:
                code_start = response.find("EXEC_CODE:") + 10
                code_str = response[code_start:].strip()
                
                # Execute the code safely
                try:
                    # Create a local copy of the DataFrame
                    local_vars = {'df': self.df.copy()}
                    exec(code_str, {}, local_vars)
                    
                    # Update the internal DataFrame
                    self.df = local_vars['df']
                    return response, self.df
                    
                except Exception as e:
                    return f"Error executing modifications: {str(e)}", None
            
            return response, None
            
        except Exception as e:
            return f"Error processing query: {str(e)}", None

# Example usage
if __name__ == "__main__":
    llm = AzureChatOpenAI(
        deployment_name="new-gpt-4o-mini",
        model_name="gpt-4o-mini",
        temperature=0
    )
    
    # Sample DataFrame
    df = pd.DataFrame({
        'A': [1, 2, 3],
        'B': [4, 5, 6]
    })
    
    agent = InteractiveDataFrameAgent(llm, df)
    
    # Example queries
    queries = [
        "Add a new column 'C' that is the sum of columns A and B",
        "What is the average of column A?",
        "Remove all rows where A is less than 2"
    ]
    
    for query in queries:
        response, modified_df = agent.process_query(query)
        print(f"\nQuery: {query}")
        print(f"Response: {response}")
        if modified_df is not None:
            print("\nModified DataFrame:")
            print(modified_df)

c:\Users\SoporteTI\Documents\project_directory\GitHubRepos\de_avatar_lowpoly\.venv\Lib\site-packages\langchain_experimental\agents\agent_toolkits\pandas\base.py:283: UserWarning: Received additional kwargs {'handle_parsing_errors': True} which are no longer supported.
  warnings.warn(




> Entering new AgentExecutor chain...
Thought: To add a new column 'C' that is the sum of columns A and B in the DataFrame `df`, I will need to modify the DataFrame by creating a new column and assigning it the result of the sum of the existing columns. 

Action: I will first calculate the sum of columns A and B and then create the new column 'C'.

Action Input: `df['C'] = df['A'] + df['B']`
I will first calculate the sum of columns A and B and then create the new column 'C'. is not a valid tool, try one of [python_repl_ast].Thought: To add a new column 'C' that is the sum of columns A and B in the DataFrame `df`, I will need to modify the DataFrame by creating a new column and assigning it the result of the sum of the existing columns. 

Action: I will execute the code to create the new column 'C'.

Action Input: `df['C'] = df['A'] + df['B']`
I will execute the code to create the new column 'C'. is not a valid tool, try one of [python_repl_ast].To add a new column 'C' that is the su

# Lllm para ejecutar código.

In [ ]:
def execute_and_return(code: str, return_vars: list = None) -> dict:
    """
    Execute code and return specified variables
    
    Args:
        code: String containing Python code to execute
        return_vars: List of variable names to return
        
    Returns:
        Dictionary containing the requested variables and their values
    """
    # Create namespace for local variables
    local_vars = {}
    
    # Execute the code
    exec(code, {}, local_vars)
    
    # If no specific variables requested, return all
    if return_vars is None:
        return local_vars
        
    # Return only requested variables
    return {var: local_vars[var] for var in return_vars if var in local_vars}

In [52]:
from langchain_openai import AzureChatOpenAI
import pandas as pd
from typing import Tuple, Optional, List, Dict, Any
import json
import re
import os
import numpy as np
from dotenv import load_dotenv
load_dotenv('../.env')


class DataFrameManipulator:
    def __init__(self, llm: AzureChatOpenAI, df: pd.DataFrame):
        """
        Initialize the DataFrameManipulator.
        
        Args:
            llm: AzureChatOpenAI instance for code generation
            df: Input DataFrame to manipulate
        """
        self.df = df
        self.llm = llm
        self.code_prompt = """Given a DataFrame 'df' and this request: "{query}"

        Write Python code to perform the requested operation. The code should:
        1. Use 'df' as the input DataFrame name
        2. Store any modified DataFrame in 'result_df'
        3. Store any analysis results in 'analysis_result'
        4. Use only pandas operations
        5. Take into account the following message from pandas: parsing datetimes with mixed time zones will raise an error unless `utc=True`
        
        The DataFrame has these columns: {columns}

        Return response in this format:
        {{
            "CODE": "<your code here>",
            "RETURN_VARS": ["result_df"] or ["analysis_result"] or ["result_df", "analysis_result"]
        }}

        Example responses:

        For modification:
        {{
            "CODE": "result_df = df.copy()\\nresult_df['new_col'] = df['A'] + df['B']",
            "RETURN_VARS": ["result_df"]
        }}

        For analysis:
        {{
            "CODE": "analysis_result = df['A'].mean()",
            "RETURN_VARS": ["analysis_result"]
        }}

        For both:
        {{
            "CODE": "result_df = df.copy()\\nresult_df['new_col'] = df['A'] + df['B']\\nanalysis_result = result_df['new_col'].mean()",
            "RETURN_VARS": ["result_df", "analysis_result"]
        }}

        Your answer will be decoded with json.loads(), therefore you must start your response with '{{' and end with '}}'.
        """
        
    def execute_and_return(self, code: str, return_vars: List[str]) -> Dict[str, Any]:
        """
        Execute code and return specified variables
        
        Args:
            code: String containing Python code to execute
            return_vars: List of variable names to return
            
        Returns:
            Dictionary containing the requested variables and their values
        """
        local_vars = {'df': self.df, 'pd': pd, 'np': np}
        
        try:
            exec(code, {'pd': pd}, local_vars)
            return {var: local_vars.get(var) for var in return_vars}
        except Exception as e:
            raise RuntimeError(f"Code execution failed: {str(e)}")

    def process_query(self, query: str) -> Tuple[str, Optional[pd.DataFrame], Optional[Any]]:
        """
        Process a natural language query to manipulate or analyze the DataFrame
        
        Args:
            query: Natural language query describing the desired operation
            
        Returns:
            Tuple containing:
            - Message describing the operation result
            - Modified DataFrame (if applicable)
            - Analysis result (if applicable)
        """
        try:
            # Get response from LLM
            response = self.llm.invoke(
                self.code_prompt.format(
                    query=query, 
                    columns=self.df.columns.tolist()
                )
            )
            
            # Parse the response
            try:
                print('LLM response:', response.content)
                instructions = json.loads(response.content)
                for key in instructions:
                    print(key,': ', instructions[key])
            except json.JSONDecodeError:
                try:
                    instructions = re.sub(r"```json.*?```", "", response.content)
                    instructions = json.loads(instructions)
                except json.JSONDecodeError:
                    raise ValueError("Invalid JSON response from LLM")
                
            if "CODE" not in instructions or "RETURN_VARS" not in instructions:
                raise ValueError("Response missing required fields")
                
            # Execute the code
            results = self.execute_and_return(
                instructions["CODE"],
                instructions["RETURN_VARS"]
            )
            
            # Process results
            result_df = results.get('result_df')
            analysis_result = results.get('analysis_result')
            
            # Update internal DataFrame if modified
            if result_df is not None and isinstance(result_df, pd.DataFrame):
                self.df = result_df
                
            # Construct return message
            message_parts = []
            if result_df is not None:
                message_parts.append("DataFrame modified")
            if analysis_result is not None:
                message_parts.append(f"Analysis result: {analysis_result}")
                
            return (
                " and ".join(message_parts),
                result_df,
                analysis_result
            )
            
        except Exception as e:
            return f"Error processing query: {str(e)}", None, None
        

df = pd.DataFrame({
    'A': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'B': [4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
})

# Example usage
llm = AzureChatOpenAI(
    deployment_name="new-gpt-4o-mini",
    model_name="gpt-4o-mini",
    temperature=0
)

manipulator = DataFrameManipulator(llm, df)

# For modification
message, modified_df, _ = manipulator.process_query("Add columns A and B into a new column C")
print('------------------------------------------------')
print('message:', message)
print('modified_df:'
      , modified_df)
print('analysis_result:', _)
print('------------------------------------------------')

# For analysis
message, _, analysis_result = manipulator.process_query("Calculate the mean of column A")
print('------------------------------------------------')
print('message:', message)
print('modified_df:'
      , _)
print('analysis_result:', analysis_result)
print('------------------------------------------------')
# For both
message, modified_df, analysis_result = manipulator.process_query("Add columns A and B, then calculate their mean")
print('------------------------------------------------')
print('message:', message)
print('modified_df:'
      , modified_df)
print('analysis_result:', analysis_result)
print('------------------------------------------------')

LLM response: {
    "CODE": "result_df = df.copy()\nresult_df['C'] = df['A'] + df['B']",
    "RETURN_VARS": ["result_df"]
}
CODE :  result_df = df.copy()
result_df['C'] = df['A'] + df['B']
RETURN_VARS :  ['result_df']
------------------------------------------------
message: DataFrame modified
modified_df:     A   B   C
0   1   4   5
1   2   5   7
2   3   6   9
3   4   7  11
4   5   8  13
5   6   9  15
6   7  10  17
7   8  11  19
8   9  12  21
9  10  13  23
analysis_result: None
------------------------------------------------
LLM response: {
    "CODE": "analysis_result = df['A'].mean()",
    "RETURN_VARS": ["analysis_result"]
}
CODE :  analysis_result = df['A'].mean()
RETURN_VARS :  ['analysis_result']
------------------------------------------------
message: Analysis result: 5.5
modified_df: None
analysis_result: 5.5
------------------------------------------------
LLM response: {
    "CODE": "result_df = df.copy()\nresult_df['sum_AB'] = df['A'] + df['B']\nanalysis_result = result_d

In [45]:
del message, modified_df, analysis_result
df = pd.DataFrame({
    'A': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'B': [4, 5, 6, 7, 8, 9, 10, 11, 12, 13],
    'CATEGORICAL': ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B']
})

manipulator = DataFrameManipulator(llm, df)
message, modified_df, analysis_result = manipulator.process_query("Make an analysis of the 'CATEGORICAL' column")
print('------------------------------------------------')
print('message:', message)
print('modified_df:'
      , modified_df)
print('analysis_result:', analysis_result)
print('------------------------------------------------')
message, modified_df, analysis_result = manipulator.process_query("Remove all rows where A is less than 5")
print('------------------------------------------------')
print('message:', message)
print('modified_df:'
      , modified_df)
print('analysis_result:', analysis_result)
print('------------------------------------------------')


LLM response: {
    "CODE": "analysis_result = df['CATEGORICAL'].value_counts()\nresult_df = df.copy()",
    "RETURN_VARS": ["result_df", "analysis_result"]
}
CODE :  analysis_result = df['CATEGORICAL'].value_counts()
result_df = df.copy()
RETURN_VARS :  ['result_df', 'analysis_result']
------------------------------------------------
message: DataFrame modified and Analysis result: CATEGORICAL
A    5
B    5
Name: count, dtype: int64
modified_df:     A   B CATEGORICAL
0   1   4           A
1   2   5           B
2   3   6           A
3   4   7           B
4   5   8           A
5   6   9           B
6   7  10           A
7   8  11           B
8   9  12           A
9  10  13           B
analysis_result: CATEGORICAL
A    5
B    5
Name: count, dtype: int64
------------------------------------------------
LLM response: {
    "CODE": "result_df = df[df['A'] >= 5]\nanalysis_result = result_df.shape[0]",
    "RETURN_VARS": ["result_df", "analysis_result"]
}
CODE :  result_df = df[df['A'] >= 5]


# Análisis con índices bursátiles (A ver si sí es cierto que mucha data science)

In [28]:
pip install yfinance

     ---------------------------------------- 0.0/948.2 kB ? eta -:--:--
     - -------------------------------------- 30.7/948.2 kB ? eta -:--:--
     - -------------------------------------- 30.7/948.2 kB ? eta -:--:--
     --- --------------------------------- 92.2/948.2 kB 871.5 kB/s eta 0:00:01
     --- --------------------------------- 92.2/948.2 kB 871.5 kB/s eta 0:00:01
     ------ ----------------------------- 163.8/948.2 kB 756.6 kB/s eta 0:00:02
     ------ ----------------------------- 163.8/948.2 kB 756.6 kB/s eta 0:00:02
     --------- -------------------------- 245.8/948.2 kB 793.0 kB/s eta 0:00:01
     --------- -------------------------- 245.8/948.2 kB 793.0 kB/s eta 0:00:01
     --------- -------------------------- 245.8/948.2 kB 793.0 kB/s eta 0:00:01
     --------- -------------------------- 245.8/948.2 kB 793.0 kB/s eta 0:00:01
     ------------- ---------------------- 368.6/948.2 kB 716.8 kB/s eta 0:00:01
     ------------- ---------------------- 368.6/948.2 kB 71


[notice] A new release of pip is available: 24.0 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# Create a ticker object for S&P 500 (^GSPC is the Yahoo Finance ticker symbol)
sp500 = yf.Ticker("^GSPC")

# Define the time period for data retrieval
end_date = datetime.today()
start_date = end_date - timedelta(days=365)  # Last 365 days of data

# Download the historical data
sp500_data = sp500.history(start=start_date, end=end_date)

# The data includes: Open, High, Low, Close, Volume, Dividends, Stock Splits
print(sp500_data.head())

# Save to CSV if needed
sp500_data.to_csv('sp500_data.csv')

                                  Open         High          Low        Close  \
Date                                                                            
2024-01-29 00:00:00-05:00  4892.950195  4929.310059  4887.399902  4927.930176   
2024-01-30 00:00:00-05:00  4925.890137  4931.089844  4916.270020  4924.970215   
2024-01-31 00:00:00-05:00  4899.189941  4906.750000  4845.149902  4845.649902   
2024-02-01 00:00:00-05:00  4861.109863  4906.970215  4853.520020  4906.189941   
2024-02-02 00:00:00-05:00  4916.060059  4975.290039  4907.990234  4958.609863   

                               Volume  Dividends  Stock Splits  
Date                                                            
2024-01-29 00:00:00-05:00  3525160000        0.0           0.0  
2024-01-30 00:00:00-05:00  3836130000        0.0           0.0  
2024-01-31 00:00:00-05:00  4696120000        0.0           0.0  
2024-02-01 00:00:00-05:00  4386090000        0.0           0.0  
2024-02-02 00:00:00-05:00  3974350000     

In [47]:
df = pd.read_csv('../assets/sp500_data.csv')
df.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2024-01-29 00:00:00-05:00,4892.950195,4929.310059,4887.399902,4927.930176,3525160000,0.0,0.0
1,2024-01-30 00:00:00-05:00,4925.890137,4931.089844,4916.270020,4924.970215,3836130000,0.0,0.0
2,2024-01-31 00:00:00-05:00,4899.189941,4906.750000,4845.149902,4845.649902,4696120000,0.0,0.0
3,2024-02-01 00:00:00-05:00,4861.109863,4906.970215,4853.520020,4906.189941,4386090000,0.0,0.0
4,2024-02-02 00:00:00-05:00,4916.060059,4975.290039,4907.990234,4958.609863,3974350000,0.0,0.0


In [48]:
result_df = df.copy()
result_df['Date'] = pd.to_datetime(result_df['Date'])
last_year = result_df[result_df['Date'] >= (result_df['Date'].max() - pd.DateOffset(years=1))]
analyis_result = last_year['Close'].mean()


C:\Users\SoporteTI\AppData\Local\Temp\ipykernel_21848\1570502129.py:2: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  result_df['Date'] = pd.to_datetime(result_df['Date'])


In [49]:
analyis_result

np.float64(5506.340559503639)

In [50]:
manipulator = DataFrameManipulator(llm, df)

# Preguntas de prueba

# 1. ¿Cuál es el precio de cierre promedio en el último año?
message, _, analysis_result = manipulator.process_query("Calculate the average closing price over the last year and return the resulting value")
print('------------------------------------------------')
print('message:', message)
print('analysis_result:', analysis_result)
print('------------------------------------------------')


# 2. ¿Cuál fue el día con el mayor volumen de transacciones?
message, _, analysis_result = manipulator.process_query("Find the date with the highest volume of transactions")
print('------------------------------------------------')
print('message:', message)
print('analysis_result:', analysis_result)
print('------------------------------------------------')


LLM response: {
    "CODE": "result_df = df.copy()\nresult_df['Date'] = pd.to_datetime(result_df['Date'])\nlast_year = result_df[result_df['Date'] >= (result_df['Date'].max() - pd.DateOffset(years=1))]\nanalysis_result = last_year['Close'].mean()",
    "RETURN_VARS": ["result_df", "analysis_result"]
}
CODE :  result_df = df.copy()
result_df['Date'] = pd.to_datetime(result_df['Date'])
last_year = result_df[result_df['Date'] >= (result_df['Date'].max() - pd.DateOffset(years=1))]
analysis_result = last_year['Close'].mean()
RETURN_VARS :  ['result_df', 'analysis_result']
------------------------------------------------
message: DataFrame modified and Analysis result: 5506.340559503639
analysis_result: 5506.340559503639
------------------------------------------------


<string>:2: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`


LLM response: {
    "CODE": "result_df = df.copy()\nanalysis_result = result_df.loc[result_df['Volume'].idxmax(), 'Date']",
    "RETURN_VARS": ["result_df", "analysis_result"]
}
CODE :  result_df = df.copy()
analysis_result = result_df.loc[result_df['Volume'].idxmax(), 'Date']
RETURN_VARS :  ['result_df', 'analysis_result']
------------------------------------------------
message: DataFrame modified and Analysis result: 2024-12-20 00:00:00-05:00
analysis_result: 2024-12-20 00:00:00-05:00
------------------------------------------------


In [37]:
message, modified_df, analysis_result = manipulator.process_query("Remove all rows where Close is less than 3000")
print('------------------------------------------------')
print('message:', message)
print('modified_df:'
      , modified_df)
print('analysis_result:', analysis_result)
print('------------------------------------------------')


LLM response: {
    "CODE": "result_df = df[df['Close'] >= 3000]\nanalyze_result = result_df.shape[0]",
    "RETURN_VARS": ["result_df", "analysis_result"]
}
CODE :  result_df = df[df['Close'] >= 3000]
analyze_result = result_df.shape[0]
RETURN_VARS :  ['result_df', 'analysis_result']
------------------------------------------------
message: DataFrame modified
modified_df:                           Date         Open         High          Low  \
0    2024-01-29 00:00:00-05:00  4892.950195  4929.310059  4887.399902   
1    2024-01-30 00:00:00-05:00  4925.890137  4931.089844  4916.270020   
2    2024-01-31 00:00:00-05:00  4899.189941  4906.750000  4845.149902   
3    2024-02-01 00:00:00-05:00  4861.109863  4906.970215  4853.520020   
4    2024-02-02 00:00:00-05:00  4916.060059  4975.290039  4907.990234   
..                         ...          ...          ...          ...   
244  2025-01-17 00:00:00-05:00  5995.399902  6014.959961  5978.439941   
245  2025-01-21 00:00:00-05:00  6014.120

In [39]:
message, modified_df, analysis_result = manipulator.process_query("Add a new column 'Daily Return' that calculates the percentage change in Close price")
print('------------------------------------------------')
print('message:', message)
print('modified_df:'
      , modified_df)
print('analysis_result:', analysis_result)
print('------------------------------------------------')


LLM response: {
    "CODE": "result_df = df.copy()\nresult_df['Daily Return'] = result_df['Close'].pct_change() * 100",
    "RETURN_VARS": ["result_df"]
}
CODE :  result_df = df.copy()
result_df['Daily Return'] = result_df['Close'].pct_change() * 100
RETURN_VARS :  ['result_df']
------------------------------------------------
message: DataFrame modified
modified_df:                           Date         Open         High          Low  \
0    2024-01-29 00:00:00-05:00  4892.950195  4929.310059  4887.399902   
1    2024-01-30 00:00:00-05:00  4925.890137  4931.089844  4916.270020   
2    2024-01-31 00:00:00-05:00  4899.189941  4906.750000  4845.149902   
3    2024-02-01 00:00:00-05:00  4861.109863  4906.970215  4853.520020   
4    2024-02-02 00:00:00-05:00  4916.060059  4975.290039  4907.990234   
..                         ...          ...          ...          ...   
244  2025-01-17 00:00:00-05:00  5995.399902  6014.959961  5978.439941   
245  2025-01-21 00:00:00-05:00  6014.120117  6

In [53]:
message, modified_df, analysis_result = manipulator.process_query("Group by month and calculate the standar deviation of the closing price")
print('------------------------------------------------')
print('message:', message)
print('modified_df:'
      , modified_df)
print('analysis_result:', analysis_result)
print('------------------------------------------------')

LLM response: {
    "CODE": "df['date'] = pd.to_datetime(df['date'], utc=True)\nresult_df = df.copy()\nresult_df.set_index('date', inplace=True)\nanalysis_result = result_df['C'].resample('M').std()",
    "RETURN_VARS": ["result_df", "analysis_result"]
}
CODE :  df['date'] = pd.to_datetime(df['date'], utc=True)
result_df = df.copy()
result_df.set_index('date', inplace=True)
analysis_result = result_df['C'].resample('M').std()
RETURN_VARS :  ['result_df', 'analysis_result']
------------------------------------------------
message: Error processing query: Code execution failed: 'date'
modified_df: None
analysis_result: None
------------------------------------------------


In [40]:
var1 = 1
var2 = 2

code = f"""
var3 = {var1} + {var2}
var4 = var3 ** 3
"""
locals = {'var1': var1, 'var2': var2}
globals = {}
exec(code, globals, locals)
var3 = locals['var3']
var4 = locals['var4']
print(var3, var4)

3 27


# Supervisor de agentes

In [31]:
# Importaciones necesarias
from langchain_openai import AzureChatOpenAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain.agents import AgentType, Tool, initialize_agent
from langchain.tools import BaseTool
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
import pandas as pd
from typing import Dict, List, Optional
from dotenv import load_dotenv
import os
import requests
import json
from pydantic import BaseModel, Field

# Cargar variables de entorno
load_dotenv('../.env')

class ForecastToolArgs(BaseModel):
    """Argumentos para la herramienta de pronóstico"""
    chronos_endpoint: str = Field(description="Endpoint para el servicio de pronóstico")
    current_df: Optional[pd.DataFrame] = Field(description="DataFrame actual para pronósticos")

    class Config:
        arbitrary_types_allowed = True

class ForecastTool(BaseTool):
    """Herramienta para generar pronósticos de series temporales."""
    
    name: str = "forecast_tool"
    description: str = """Útil para generar pronósticos de series temporales.
    Requiere un JSON con el siguiente formato:
    {
        "target_column": "nombre_columna",  # Columna objetivo para el pronóstico
        "prediction_length": 30             # Número de períodos a predecir (opcional, default 30)
    }
    Ejemplo de uso: {"target_column": "ventas", "prediction_length": 30}
    Devuelve un DataFrame con las predicciones y los intervalos de confianza."""
    args_schema: type[BaseModel] = ForecastToolArgs
    
    def __init__(self, chronos_endpoint: str):
        super().__init__()
        self._chronos_endpoint = chronos_endpoint
        # current_df = None
        self._current_df = None
    
    async def generate_forecast(self, target_column: str, prediction_length: int = 30) -> Dict:
        """Generate forecasts using the Chronos endpoint"""
        if self._current_df is None:
            return {"error": "No data loaded"}

        try:
            # Prepare CSV data
            csv_data = self._current_df.to_csv(index=False)
            
            # Prepare the request
            files = {
                'file': ('./assets/orders_example.csv', csv_data, 'text/csv')
            }
            data = {
                'prediction_length': str(prediction_length),
                'target_column': target_column
            }
            response = requests.post(self._chronos_endpoint, data=data, files=files)
            if response.status_code == 200:
                last_forecast = response.json()

                # Extend the target column with forecast values
                extended_values = last_forecast['actual'] + last_forecast['forecast']
                lower_quantile = [None for _ in range(len(last_forecast['actual']))] + last_forecast['lower_quantile']
                upper_quantile = [None for _ in range(len(last_forecast['actual']))] + last_forecast['upper_quantile']
                future_assigned = [False] * len(last_forecast['actual']) + [True] * len(last_forecast['forecast'])

                # Update the DataFrame
                # Get the periodicity of the first column
                dates = pd.to_datetime(self._current_df[self._current_df.columns.tolist()[0]])
                freq = pd.infer_freq(dates)

                # Generate future dates
                future_dates = pd.date_range(start=dates.iloc[-1], periods=prediction_length + 1, freq=freq)[1:]

                # Extend the DataFrame with future dates
                self._current_df = pd.concat([
                    self._current_df, 
                    pd.DataFrame({self._current_df.columns.tolist()[0]: future_dates})
                ]).reset_index(drop=True)
                self._current_df[target_column] = pd.Series(extended_values[:len(self._current_df)])
                self._current_df['future_assigned_column'] = pd.Series(future_assigned[:len(self._current_df)])
                self._current_df['lower_quantile'] = pd.Series(lower_quantile[:len(self._current_df)])
                self._current_df['upper_quantile'] = pd.Series(upper_quantile[:len(self._current_df)])
                return self._current_df
            else:
                return {"error": f"Forecast error: {response.text}"}
        except Exception as e:
            return {"error": f"Error generating forecast: {str(e)}"}
        
    def _run(self, args: str) -> Dict:
        """Ejecuta el pronóstico basado en los argumentos proporcionados"""
        try:
            # Parsear los argumentos (asumiendo formato JSON)
            params = json.loads(args)
            target_column = params.get('target_column')
            prediction_length = params.get('prediction_length', 30)
            
            if not target_column:
                return {"error": "target_column is required"}
            
            # Llamar a la función de pronóstico usando asyncio
            import asyncio
            return asyncio.run(self.generate_forecast(target_column, prediction_length))
        except Exception as e:
            return {"error": f"Error in forecast tool: {str(e)}"}
            
    def _arun(self, args: str):
        raise NotImplementedError("Async not implemented")

class AnalysisOrchestrator:
    def __init__(self, azure_deployment: str, chronos_endpoint: str):
        # Inicializar el modelo LLM
        self.llm = AzureChatOpenAI(
            deployment_name=azure_deployment,
            model_name="gpt-4o-mini",
            temperature=0
        )
        
        # Inicializar el DataFrame
        self.df = None
        
        # Crear las herramientas
        self.forecast_tool = ForecastTool(chronos_endpoint=chronos_endpoint)
        
        # Definir el prompt para el agente supervisor
        self.supervisor_prompt = PromptTemplate(
            input_variables=["input"],
            template="""Eres un asistente experto en análisis de datos y forecasting.
            Basado en la siguiente consulta, determina si necesitas realizar análisis de datos,
            generar pronósticos, o ambos. Luego, coordina las herramientas apropiadas para resolver la tarea.

            Consulta: {input}

            Por favor:
            1. Analiza la consulta y determina las subtareas necesarias
            2. Ejecuta las subtareas en el orden correcto
            3. Combina los resultados en una respuesta coherente

            Recuerda:
            - Para análisis de datos, usa el pandas_agent
            - Para pronósticos, usa el forecast_tool
            - Puedes combinar resultados de ambas herramientas si es necesario
            - No es necesario importar los datos, ya están instanciados en el objeto df
            """
        )
    
    def load_data(self, df: pd.DataFrame):
        """Carga un DataFrame para análisis"""
        self.df = df
        # Actualizar el PandasAgent con el nuevo DataFrame
        self.pandas_agent = create_pandas_dataframe_agent(
            self.llm,
            df =  self.df,
            verbose=True,
            handling_parsing_errors=True,
            agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
            handle_parsing_errors=True,
            max_iterations=6,
            allow_dangerous_code=True
        )
        # Actualizar el ForecastTool con el nuevo DataFrame
        self.forecast_tool._current_df = self.df
    
    def create_supervisor_agent(self):
        """Crea el agente supervisor con todas las herramientas disponibles"""
        tools = [
            Tool(
                name="pandas_agent",
                func=self.pandas_agent.run,
                description="Útil para análisis de datos y manipulación de DataFrames"
            ),
            Tool(
                name="forecast_tool",
                func=self.forecast_tool._run,
                description="Útil para generar pronósticos de series temporales"
            )
        ]
        
        return initialize_agent(
            tools,
            self.llm,
            agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
            verbose=True
        )
    
    def process_query(self, query: str) -> Dict:
        """Procesa una consulta utilizando el agente supervisor"""
        if self.df is None:
            return {"error": "No data loaded"}
            
        try:
            supervisor = self.create_supervisor_agent()
            result = supervisor.run(self.supervisor_prompt.format(input=query))
            return {"result": result}
        except Exception as e:
            return {"error": f"Error processing query: {str(e)}"}

# Ejemplo de uso
# if __name__ == "__main__":
    # Crear el orquestador
orchestrator = AnalysisOrchestrator(
    azure_deployment=os.getenv("DEPLOYMENT_NAME"),
    chronos_endpoint="https://localhost:8000/forecast"
)

# Cargar datos de ejemplo
df = pd.DataFrame({
    'fecha': pd.date_range(start='2023-01-01', periods=100),
    'ventas': range(100),
    'categoria': ['A', 'B'] * 50
})
orchestrator.load_data(df)

# Ejemplos de consultas
queries = [
    # "¿Cuál es el promedio de ventas por categoría?",
    "Genera un pronóstico de ventas para los próximos 30 días",
    "Analiza las ventas por categoría y genera un pronóstico para la categoría A"
]

for query in queries:
    print(f"\nConsulta: {query}")
    result = orchestrator.process_query(query)
    print("Resultado:", result)

c:\Users\SoporteTI\Documents\project_directory\GitHubRepos\de_avatar_lowpoly\.venv\Lib\site-packages\langchain_experimental\agents\agent_toolkits\pandas\base.py:283: UserWarning: Received additional kwargs {'handling_parsing_errors': True, 'handle_parsing_errors': True} which are no longer supported.
  warnings.warn(



Consulta: ¿Cuál es el promedio de ventas por categoría?


> Entering new AgentExecutor chain...
Necesitamos realizar análisis de datos para calcular el promedio de ventas por categoría. Primero, debemos agrupar los datos por categoría y luego calcular el promedio de ventas para cada categoría.
Action: pandas_agent
Action Input: 
```
df.groupby('category')['sales'].mean()
```

> Entering new AgentExecutor chain...
Thought: We want to group the dataframe by category and calculate the mean of sales for each category.
Action: python_repl_ast
Action Input: `df.groupby('category')['sales'].mean()`KeyError: 'category'The error message suggests that there is no column named 'category' in the dataframe. We should check the column names to make sure we are using the correct one.
Action: python_repl_ast
Action Input: `df.columns`Index(['fecha', 'ventas', 'categoria'], dtype='object')The column name for category is actually 'categoria'. We should use that instead.
Action: python_repl_ast
Action I

<string>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


We have filtered the dataframe to only include rows where the categoria column is equal to "B" and converted the fecha column to the correct format for the forecast_tool. Now we need to create the input data for the forecast_tool.
Action: python_repl_ast
Action Input:
```
input_data = {'start': df_b['fecha'].iloc[0], 'freq': 'D', 'target': df_b['ventas'].tolist()}
input_data
```{'start': '2023-01-02', 'freq': 'D', 'target': [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41, 43, 45, 47, 49, 51, 53, 55, 57, 59, 61, 63, 65, 67, 69, 71, 73, 75, 77, 79, 81, 83, 85, 87, 89, 91, 93, 95, 97, 99]}We have successfully created the input data for the forecast_tool. 
Final Answer: {'start': '2023-01-02', 'freq': 'D', 'target': [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41, 43, 45, 47, 49, 51, 53, 55, 57, 59, 61, 63, 65, 67, 69, 71, 73, 75, 77, 79, 81, 83, 85, 87, 89, 91, 93, 95, 97, 99]}

> Finished chain.

Observation: {'start': '2023-0

In [82]:
# Importaciones necesarias
from langchain_openai import AzureChatOpenAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain.agents import AgentType, Tool, initialize_agent
from langchain.tools import BaseTool
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
import pandas as pd
from typing import Dict, List, Optional
from dotenv import load_dotenv
import os
import requests
import json
from pydantic import BaseModel, Field

# Cargar variables de entorno
load_dotenv('../.env')

class ForecastToolArgs(BaseModel):
    """Argumentos para la herramienta de pronóstico"""
    chronos_endpoint: str = Field(description="Endpoint para el servicio de pronóstico")
    current_df: Optional[pd.DataFrame] = Field(description="DataFrame actual para pronósticos")

    class Config:
        arbitrary_types_allowed = True

class ForecastTool(BaseTool):
    """Herramienta para generar pronósticos de series temporales."""
    
    name: str = "forecast_tool"
    description: str = """Útil para generar pronósticos de series temporales.
    Requiere un JSON con el siguiente formato:
    {
        "target_column": "nombre_columna",  # Columna objetivo para el pronóstico
        "prediction_length": 30             # Número de períodos a predecir (opcional, default 30)
    }
    Ejemplo de uso: {"target_column": "ventas", "prediction_length": 30}
    Devuelve un DataFrame con las predicciones y los intervalos de confianza."""
    args_schema: type[BaseModel] = ForecastToolArgs
    
    def __init__(self, chronos_endpoint: str):
        super().__init__()
        self._chronos_endpoint = chronos_endpoint
        # current_df = None
        self._current_df = None
    
    def _run(self, args: str) -> Dict:
        """Ejecuta el pronóstico basado en los argumentos proporcionados de forma síncrona"""
        try:
            # Parsear los argumentos
            params = json.loads(args)
            target_column = params.get('target_column')
            prediction_length = params.get('prediction_length', 30)
            
            if not target_column:
                return {"error": "target_column is required"}

            if self._current_df is None:
                return {"error": "No data loaded"}

            try:
                # Prepare CSV data
                csv_data = self._current_df.to_csv(index=False)
                
                # Prepare the request
                files = {
                    'file': ('../assets/orders_example.csv', csv_data, 'text/csv')
                }
                data = {
                    'prediction_length': prediction_length,
                    'target_column': target_column
                }
                response = requests.post(self._chronos_endpoint, data=data, files=files)
                if response.status_code == 200:
                    last_forecast = response.json()

                    # Extend the target column with forecast values
                    extended_values = last_forecast['actual'] + last_forecast['forecast']
                    lower_quantile = [None for _ in range(len(last_forecast['actual']))] + last_forecast['lower_quantile']
                    upper_quantile = [None for _ in range(len(last_forecast['actual']))] + last_forecast['upper_quantile']
                    future_assigned = [False] * len(last_forecast['actual']) + [True] * len(last_forecast['forecast'])

                    # Update the DataFrame
                    # Get the periodicity of the first column
                    dates = pd.to_datetime(self._current_df[self._current_df.columns.tolist()[0]])
                    freq = pd.infer_freq(dates)

                    # Generate future dates
                    future_dates = pd.date_range(start=dates.iloc[-1], periods=prediction_length + 1, freq=freq)[1:]

                    # Extend the DataFrame with future dates
                    self._current_df = pd.concat([
                        self._current_df, 
                        pd.DataFrame({self._current_df.columns.tolist()[0]: future_dates})
                    ]).reset_index(drop=True)
                    self._current_df[target_column] = pd.Series(extended_values[:len(self._current_df)])
                    self._current_df['future_assigned_column'] = pd.Series(future_assigned[:len(self._current_df)])
                    self._current_df['lower_quantile'] = pd.Series(lower_quantile[:len(self._current_df)])
                    self._current_df['upper_quantile'] = pd.Series(upper_quantile[:len(self._current_df)])
                    return self._current_df
                else:
                    return {"error": f"Forecast error: {response.text}"}
            except Exception as e:
                return {"error": f"Error generating forecast: {str(e)}"}
                
        except Exception as e:
            return {"error": f"Error in forecast tool: {str(e)}"}
    
    def _arun(self, args: str):
        raise NotImplementedError("Async not implemented")

class AnalysisOrchestrator:
    def __init__(self, azure_deployment: str, chronos_endpoint: str):
        # Inicializar el modelo LLM
        self.llm = AzureChatOpenAI(
            deployment_name=azure_deployment,
            model_name="gpt-4o-mini",
            temperature=0
        )
        
        # Inicializar el DataFrame
        self.df = None
        
        # Crear las herramientas
        self.forecast_tool = ForecastTool(chronos_endpoint=chronos_endpoint)
        
        # Definir el prompt para el agente supervisor
        self.supervisor_prompt = PromptTemplate(
            input_variables=["input"],
            template="""You are an expert assistant in data analysis and forecasting.
            Based on the following query, determine if you need to perform data analysis,
            generate forecasts, or both. Then, coordinate the appropriate tools to solve the task.

            Query: {input}

            Please:
            1. Analyze the query and determine the required task
            2. Execute the subtasks in the correct order
            3. Combine the results into a coherent response

            Remember:
            - For data analysis, use the pandas_agent
            - For forecasts, use the forecast_tool
            - You can combine results from both tools if necessary
            - It is not necessary to import the data, it is already instantiated in the df object
            - You can only send one task at a time, do not send multiple tasks in a single query
            - The input format for the forecast_tool is a JSON in the format: {{"target_column": "column_name", "prediction_length": 30}}
            - For Action you just provide the tool name 
            - For Action Input you just provide the input for the tool
            """
        )
    
    def load_data(self, df: pd.DataFrame):
        """Carga un DataFrame para análisis"""
        self.df = df
        # Actualizar el PandasAgent con el nuevo DataFrame
        self.pandas_agent = create_pandas_dataframe_agent(
            self.llm,
            df =  self.df,
            verbose=True,
            agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
            handle_parsing_errors=True,
            max_iterations=6,
            allow_dangerous_code=True
        )
        # Actualizar el ForecastTool con el nuevo DataFrame
        self.forecast_tool._current_df = self.df
    
    def create_supervisor_agent(self):
        """Crea el agente supervisor con todas las herramientas disponibles"""
        class ForecasterInput(BaseModel):
            input: str = Field(description="A string in JSON format, with the following structure:  {'target_column': 'column_name', 'prediction_length': 30}")
        tools = [
            Tool(
                name="pandas_agent",
                func=self.pandas_agent.run,
                description="Useful for data analysis and DataFrame manipulation",
            ),
            Tool(
                name="forecast_tool",
                func=self.forecast_tool._run,
                description="Useful for generating time series forecasts",
                input=ForecasterInput
            )
        ]
        
        return initialize_agent(
            tools,
            self.llm,
            agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
            handle_parsing_errors=True,
            verbose=True
        )
    
    def process_query(self, query: str) -> Dict:
        """Procesa una consulta utilizando el agente supervisor"""
        if self.df is None:
            return {"error": "No data loaded"}
            
        try:
            supervisor = self.create_supervisor_agent()
            result = supervisor.run(self.supervisor_prompt.format(input=query))
            return {"result": result}
        except Exception as e:
            return {"error": f"Error processing query: {str(e)}"}

# Ejemplo de uso
# if __name__ == "__main__":
    # Crear el orquestador
orchestrator = AnalysisOrchestrator(
    azure_deployment=os.getenv("DEPLOYMENT_NAME"),
    chronos_endpoint="http://localhost:8000/forecast"
)

# Cargar datos de ejemplo
df = pd.DataFrame({
    'fecha': pd.date_range(start='2023-01-01', periods=100),
    'ventas': range(100),
    'categoria': ['A', 'B'] * 50
})
orchestrator.load_data(df)

# Ejemplos de consultas
queries = [
    # "¿Cuál es el promedio de ventas por categoría?",
    "Generate a \"ventas\" forecast for the next 30 days. Verify the columns names first.",
    "Analyse \"ventas\" by category and generate a forecast for category A. Verify the column names first"
]

for query in queries:
    print(f"\nConsulta: {query}")
    result = orchestrator.process_query(query)
    print("Resultado:", result)

c:\Users\SoporteTI\Documents\project_directory\GitHubRepos\de_avatar_lowpoly\.venv\Lib\site-packages\langchain_experimental\agents\agent_toolkits\pandas\base.py:283: UserWarning: Received additional kwargs {'handle_parsing_errors': True} which are no longer supported.
  warnings.warn(



Consulta: Generate a "ventas" forecast for the next 30 days. Verify the columns names first.


> Entering new AgentExecutor chain...
The query requires generating a forecast for the "ventas" column for the next 30 days. We need to verify the column names first, which suggests we may need to perform some data analysis before generating the forecast. 

Action: pandas_agent
Action Input: df.columns

> Entering new AgentExecutor chain...
Thought: We want to see the column names of the dataframe
Action: python_repl_ast
Action Input: df.columnsIndex(['fecha', 'ventas', 'categoria'], dtype='object')We now know the column names of the dataframe
Final Answer: ['fecha', 'ventas', 'categoria']

> Finished chain.

Observation: ['fecha', 'ventas', 'categoria']
Thought:We have verified that the column names are correct and that we have a "ventas" column. Now we can move on to generating the forecast.

Action: forecast_tool
Action Input: {"target_column": "ventas", "prediction_length": 30}
Observati

In [44]:
help(Tool)

Help on class Tool in module langchain_core.tools.simple:

class Tool(langchain_core.tools.base.BaseTool)
 |  Tool(name: 'str', func: 'Optional[Callable]', description: 'str', *, args_schema: Annotated[Optional[type[pydantic.main.BaseModel]], SkipValidation()] = None, return_direct: bool = False, verbose: bool = False, callbacks: Union[list[langchain_core.callbacks.base.BaseCallbackHandler], langchain_core.callbacks.base.BaseCallbackManager, NoneType] = None, callback_manager: Optional[langchain_core.callbacks.base.BaseCallbackManager] = None, tags: Optional[list[str]] = None, metadata: Optional[dict[str, Any]] = None, handle_tool_error: Union[bool, str, Callable[[langchain_core.tools.base.ToolException], str], NoneType] = False, handle_validation_error: Union[bool, str, Callable[[Union[pydantic_core._pydantic_core.ValidationError, pydantic.v1.error_wrappers.ValidationError]], str], NoneType] = False, response_format: Literal['content', 'content_and_artifact'] = 'content', coroutine: O